# Fortran 代码学习与复现 Notebook

本 Notebook 用于小范围学习和复现 MR_CDFT 项目中的 Fortran 代码。

## 环境设置
- 已安装 `fortran-magic`
- 编译器：`gfortran` (GCC 13.2.0)
- 项目路径：`code/src/`

In [ ]:
# 加载 fortran-magic 扩展
%load_ext fortranmagic
import numpy as np
import matplotlib.pyplot as plt

print("Fortran magic loaded!")

## 1. 基础示例：Fortran 子程序

先从简单的 Fortran 代码开始，熟悉 `fortran-magic` 的使用。

In [ ]:
%%fortran
! 计算原子核的半径 (液滴模型)
subroutine nuclear_radius(A, r0, R)
    implicit none
    real(8), intent(in) :: A    ! 质量数
    real(8), intent(in) :: r0   ! 半径参数 (fm)
    real(8), intent(out) :: R   ! 半径 (fm)
    
    R = r0 * A**(1.0d0/3.0d0)
end subroutine nuclear_radius

! 计算结合能 (Bethe-Weizsäcker 公式)
subroutine binding_energy(A, Z, B)
    implicit none
    real(8), intent(in) :: A, Z
    real(8), intent(out) :: B
    real(8) :: a_v, a_s, a_c, a_a, delta
    
    ! 系数 (MeV)
    a_v = 15.8d0  ! 体积项
    a_s = 18.3d0  ! 表面项
    a_c = 0.714d0 ! 库仑项
    a_a = 23.2d0  ! 不对称项
    
    ! 配对项
    if (mod(int(A), 2) == 0 .and. mod(int(Z), 2) == 0) then
        delta = 12.0d0 / sqrt(A)
    else if (mod(int(A), 2) == 1) then
        delta = 0.0d0
    else
        delta = -12.0d0 / sqrt(A)
    end if
    
    B = a_v*A - a_s*A**(2.0d0/3.0d0) - a_c*Z*(Z-1)/A**(1.0d0/3.0d0) &
        - a_a*(A-2*Z)**2/A + delta
end subroutine binding_energy

In [ ]:
# 测试 Fortran 子程序
import numpy as np

# 计算 ^208Pb 的半径
A = 208.0
Z = 82.0
r0 = 1.2  # fm
R = 0.0

# 调用 Fortran 子程序
fortran.nuclear_radius(A, r0, R)
print(f"^208Pb 的半径: {R:.2f} fm")

# 计算结合能
B = 0.0
fortran.binding_energy(A, Z, B)
print(f"^208Pb 的结合能: {B:.2f} MeV")
print(f"每个核子的结合能: {B/A:.2f} MeV/A")

## 2. 复现项目中的物理常数

从 `Constants.f90` 中提取物理常数，并在 Fortran 中使用。

In [ ]:
%%fortran
module PhysicsConstants
    implicit none
    real(8), parameter :: pi = 3.141592653589793d0
    real(8), parameter :: hbar_c = 197.328284d0  ! hbar*c (MeV·fm)
    real(8), parameter :: r0_default = 1.2d0     ! 半径参数 (fm)
    real(8), parameter :: alpha = 1.0d0/137.03602d0  ! 精细结构常数
contains
    
    ! 计算约化康普顿波长
    real(8) function compton_wavelength(mass)
        real(8), intent(in) :: mass  ! 质量 (MeV/c^2)
        compton_wavelength = hbar_c / mass  ! (fm)
    end function compton_wavelength
    
    ! 计算库仑势 (点电荷)
    real(8) function coulomb_potential(r, Z1, Z2)
        real(8), intent(in) :: r   ! 距离 (fm)
        integer, intent(in) :: Z1, Z2  ! 电荷数
        coulomb_potential = alpha * hbar_c * Z1 * Z2 / r  ! (MeV)
    end function coulomb_potential
end module PhysicsConstants

In [ ]:
# 测试物理常数模块
from fortran import PhysicsConstants

print("物理常数:")
print(f"  pi = {PhysicsConstants.pi}")
print(f"  hbar*c = {PhysicsConstants.hbar_c} MeV·fm")
print(f"  精细结构常数 alpha = {PhysicsConstants.alpha}")

# 计算质子和中子的康普顿波长
m_p = 938.272d0  # 质子质量 (MeV/c^2)
m_n = 939.565d0  # 中子质量 (MeV/c^2)

lambda_p = PhysicsConstants.compton_wavelength(m_p)
lambda_n = PhysicsConstants.compton_wavelength(m_n)

print(f"\n质子康普顿波长: {lambda_p:.4f} fm")
print(f"中子康普顿波长: {lambda_n:.4f} fm")

# 计算两个质子在 1 fm 距离的库仑势
V_coul = PhysicsConstants.coulomb_potential(1.0d0, 1, 1)
print(f"\n两个质子的库仑势 (r=1 fm): {V_coul:.4f} MeV")

## 3. 数值方法：高斯积分

从 `Mathmethods.f90` 中提取数值方法。
高斯积分是核物理计算中常用的数值积分方法。

In [ ]:
%%fortran
module GaussianQuadrature
    implicit none
    integer, parameter :: max_points = 32
contains
    
    ! 高斯-勒让德积分的节点和权重 (低阶)
    subroutine gauss_legendre_8(x, w)
        real(8), intent(out) :: x(8), w(8)
        ! 8点高斯-勒让德积分的节点和权重
        x(1) = -0.960289856497536d0; w(1) = 0.101228536290377d0
        x(2) = -0.796666477413627d0; w(2) = 0.222381034453375d0
        x(3) = -0.525532409916329d0; w(3) = 0.313706645877887d0
        x(4) = -0.183434642495650d0; w(4) = 0.362683783378362d0
        x(5) =  0.183434642495650d0; w(5) = 0.362683783378362d0
        x(6) =  0.525532409916329d0; w(6) = 0.313706645877887d0
        x(7) =  0.796666477413627d0; w(7) = 0.222381034453375d0
        x(8) =  0.960289856497536d0; w(8) = 0.101228536290377d0
    end subroutine gauss_legendre_8
    
    ! 使用高斯-勒让德积分计算 ∫f(x)dx from -1 to 1
    real(8) function gauss_integrate(f, n)
        interface
            real(8) function f(x)
                real(8), intent(in) :: x
            end function f
        end interface
        integer, intent(in) :: n
        real(8) :: x(32), w(32)
        integer :: i
        
        gauss_integrate = 0.0d0
        if (n == 8) then
            call gauss_legendre_8(x, w)
            do i = 1, 8
                gauss_integrate = gauss_integrate + w(i) * f(x(i))
            end do
        end if
    end function gauss_integrate
    
    ! 变换到任意区间 [a, b]
    real(8) function gauss_integrate_ab(f, a, b, n)
        interface
            real(8) function f(x)
                real(8), intent(in) :: x
            end function f
        end interface
        real(8), intent(in) :: a, b
        integer, intent(in) :: n
        real(8) :: scale, shift
        
        scale = (b - a) / 2.0d0
        shift = (a + b) / 2.0d0
        
        ! 定义变换后的函数
        gauss_integrate_ab = scale * gauss_integrate(f_transformed, n)
    contains
        real(8) function f_transformed(xi)
            real(8), intent(in) :: xi
            f_transformed = f(scale * xi + shift)
        end function f_transformed
    end function gauss_integrate_ab
end module GaussianQuadrature

In [ ]:
# 测试高斯积分
from fortran import GaussianQuadrature
import numpy as np

# 定义测试函数
def f1(x):
    return x**2

def f2(x):
    return np.exp(x)

def f3(x):
    return 1.0 / (1.0 + x**2)

# 测试 ∫x^2 dx from -1 to 1 (解析解: 2/3)
result1 = GaussianQuadrature.gauss_integrate(f1, 8)
print(f"∫x^2 dx from -1 to 1:")
print(f"  数值解: {result1:.10f}")
print(f"  解析解: {2.0/3.0:.10f}")
print(f"  误差: {abs(result1 - 2.0/3.0):.2e}")

# 测试 ∫e^x dx from 0 to 1 (解析解: e-1)
result2 = GaussianQuadrature.gauss_integrate_ab(f2, 0.0, 1.0, 8)
print(f"\n∫e^x dx from 0 to 1:")
print(f"  数值解: {result2:.10f}")
print(f"  解析解: {np.exp(1) - 1:.10f}")
print(f"  误差: {abs(result2 - (np.exp(1) - 1)):.2e}")

## 4. 简单核物质密度分布

复现项目中的密度计算（简化版）。
参考 `CDFT_density.f90` 和 `Proj_density.f90`。

In [ ]:
%%fortran
module NuclearDensity
    use PhysicsConstants, only: pi, r0_default
    implicit none
contains
    
    ! 均匀球模型密度
    real(8) function uniform_density(r, A)
        real(8), intent(in) :: r  ! 径向坐标 (fm)
        integer, intent(in) :: A  ! 质量数
        real(8) :: R, rho0
        
        R = r0_default * A**(1.0d0/3.0d0)  ! 核半径
        rho0 = 3.0d0 * A / (4.0d0 * pi * R**3)  ! 归一化密度
        
        if (r <= R) then
            uniform_density = rho0
        else
            uniform_density = 0.0d0
        end if
    end function uniform_density
    
    ! 费米分布密度 (Fermi function)
    real(8) function fermi_density(r, A, a)
        real(8), intent(in) :: r   ! 径向坐标 (fm)
        integer, intent(in) :: A    ! 质量数
        real(8), intent(in) :: a    ! 扩散宽度 (fm)
        real(8) :: R, rho0
        
        R = r0_default * A**(1.0d0/3.0d0)
        rho0 = 3.0d0 * A / (4.0d0 * pi * R**3)  ! 近似归一化
        
        fermi_density = rho0 / (1.0d0 + exp((r - R) / a))
    end function fermi_density
    
    ! 谐振子基密度 (简化版)
    real(8) function harmonic_oscillator_density(r, A, b)
        real(8), intent(in) :: r  ! 径向坐标 (fm)
        integer, intent(in) :: A    ! 质量数
        real(8), intent(in) :: b    ! 谐振子长度 (fm)
        real(8) :: rho0, x
        
        x = r / b
        rho0 = 2.0d0 * A / (b**3 * pi**1.5d0)  ! 近似归一化
        harmonic_oscillator_density = rho0 * exp(-x**2)
    end function harmonic_oscillator_density
end module NuclearDensity

In [ ]:
# 可视化密度分布
from fortran import NuclearDensity
import numpy as np
import matplotlib.pyplot as plt

# 设置核素
A = 208  # ^208Pb
Z = 82

# 径向坐标
r = np.linspace(0, 10, 200)

# 计算密度
rho_uniform = np.array([NuclearDensity.uniform_density(ri, A) for ri in r])
rho_fermi = np.array([NuclearDensity.fermi_density(ri, A, 0.5) for ri in r])
rho_ho = np.array([NuclearDensity.harmonic_oscillator_density(ri, A, 2.5) for ri in r])

# 绘图
plt.figure(figsize=(10, 6))
plt.plot(r, rho_uniform, label='均匀球模型', linewidth=2)
plt.plot(r, rho_fermi, label='费米分布 (a=0.5 fm)', linewidth=2)
plt.plot(r, rho_ho, label='谐振子分布 (b=2.5 fm)', linewidth=2)
plt.xlabel('r (fm)')
plt.ylabel('ρ(r) (fm$^{-3}$)')
plt.title(f'^208Pb 密度分布')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. 下一步学习建议

1. **阅读项目代码**：从 `CDFT_main.f90` 开始，理解主流程
2. **复现简单模块**：
   - `Mathmethods.f90`：数值方法
   - `CDFT_density.f90`：密度计算
   - `CDFT_expectation.f90`：期望值计算
3. **使用 Python 重新实现**：用 NumPy 实现相同功能，对比结果
4. **逐步增加复杂度**：从简单模型到完整 CDFT 计算